#Architecture
Data Loader  
Signal Detection  
Risk Models  
Risk Engine  
Evaluation  
Simulator


In [ ]:
# Importing Dependencies
import pandas as pd
import numpy as np


#Data Loader

In [ ]:
def load_event_data(path):
    return pd.read_csv(path)


 # Signal Detection
 This is the first signal detector turning raw counts into statistical emaning using z-score to measure how unusual an observed adverse event count is compared to what’s expected. Assumption: data is roughly normally distributed.

observed: actual number of adverse events (e.g., 14 cases today)
  
  expected: baseline or average (e.g., normally 6 cases)
  
  std: standard deviation, how much the data fluctuates

In [ ]:
def z_score_signal(observed, expected, std):
    return (observed - expected) / std


#Risk Models
 This is where the system starts thinking probabilistically after detecting signal anomalies. Here, the system updates risk beliefs using Bayesian thinking (starts with a baseline risk belief and adjusts it gradually as new signals come). It also includes temporal weighting: for giving more importance to recent data.









In [ ]:
def bayesian_update(prior, likelihood):
    return (prior * likelihood) / ((prior * likelihood) + (1 - prior) * (1 - likelihood))

def temporal_weighting(events, decay=0.9):
    weights = [decay**i for i in range(len(events))]
    return np.dot(events, weights) / sum(weights)

# Evaluation

Up until this point, the system detected anomalies (Z-score), updated belief (Bayesian) and captured trends (temporal weighting). Here it combines signals into a single risk score to answer "should we be concerened?"
  
  **freq_score**: how frequently adverse events are happening


   **severity**: how serious the events are
   
   
   **uncertainty**: Comes from the Bayesian update (posterior probability), how confident we are that this drug is risky

  *this is a weighted sum where frequency matters most followed by severity and statistical confidence.
  * Note the weights are manually chosen


In [ ]:
def composite_risk_score(freq_score, severity, uncertainty):
    return (0.5 * freq_score) + (0.3 * severity) + (0.2 * uncertainty)

#Risk Engine

This is where the system transitions from individual calculations to a decision-making system




In [ ]:

def evaluate_risk(event_data, baseline, std, severity, prior):
    z = z_score_signal(event_data[-1], baseline, std)
    likelihood = 1 / (1 + abs(z))
    posterior = bayesian_update(prior, likelihood)
    trend = temporal_weighting(event_data)

    return composite_risk_score(trend, severity, posterior)


#Implementation
Running the system using recent adverse event counts over time

In [ ]:


data = [5, 7, 6, 9, 14]
risk = evaluate_risk(data, baseline=6, std=2, severity=0.8, prior=0.3)

print(f"Final Risk Score: {risk:.3f}")


Final Risk Score: 4.155


#Simulated Adverse Event Dataset

| Column          | Description                                |
| --------------- | ------------------------------------------ |
| patient_id      | Unique patient identifier                  |
| drug            | Drug name                                  |
| adverse_event   | Type of adverse event                      |
| severity        | Normalized severity score (0–1)            |
| report_date     | Date of event                              |
| baseline_rate   | Expected event frequency                   |
| observed_events | Events observed for that drug on that date |


# Data Simulator
Creating pharmacovigilance data that mimics how adverse events are reported continuously over time that monitors different drugs and different types of adverse events.
Expected output: drug | date | observed_events | baseline_rate | severity

In [ ]:
def generate_adverse_event_data(n_patients=5000, days=120):
    np.random.seed(42)

    drugs = ["DrugA", "DrugB", "DrugC"]
    events = ["Nausea", "Arrhythmia", "Fatigue", "Headache"]

    rows = []
    start_date = pd.to_datetime("2025-01-01")

    for day in range(days):
        for _ in range(n_patients):
            patient_id = np.random.randint(100000, 999999)
            drug = np.random.choice(drugs, p=[0.4, 0.35, 0.25])
            event = np.random.choice(events)

            baseline = np.random.uniform(2, 6)
            observed = np.random.poisson(baseline)

            severity = np.clip(np.random.normal(0.6, 0.2), 0, 1)
            date = start_date + pd.Timedelta(days=day)

            rows.append([
                patient_id, drug, event, severity,
                date, baseline, observed
            ])

    return pd.DataFrame(rows, columns=[
        "patient_id", "drug", "adverse_event", "severity",
        "report_date", "baseline_rate", "observed_events"
    ])